# Somo 11 - Itifaki ya Wakala kwa Wakala (A2A)


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Itifaki ya A2A ni Nini?

The **Itifaki ya Wakala kwa Wakala (A2A)** ni kiwango wazi kinachowezesha maajenti za AI kuwasiliana,
kugundua kila mmoja, na kushirikiana — hata wakati zimetengenezwa kwa mifumo tofauti au zikiwa zimehifadhiwa
na huduma tofauti.

Key concepts:

- **Ugunduzi** – Mawakala huchapisha *Kadi ya Wakala* inayofafanua uwezo wao, ikifanya iwe rahisi kwa mawakala wengine (au mratibu) kupata mtaalamu sahihi kwa kazi.
- **Kupitisha Ujumbe** – Mawakala hubadilishana ujumbe wenye muundo kupitia itifaki ya pamoja, hivyo ombi kutoka kwa wakala mmoja linaweza kueleweka na kutimizwa na mwingine bila kujali utekelezaji wake wa ndani.
- **Mzunguko wa Kazi** – A2A inafafanua hali kama *imetumwa*, *inayofanya kazi*, *imetimizwa*, na *imeshindwa*, na kumpa mratibu uwazi kamili kuhusu jinsi kazi iliyoteuliwa inavyoendelea.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Kuunda Mawakala Maalum wa Usafiri


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Ushirikiano wa Mawakala Wengi kupitia Mtiririko wa Kazi

Tunawaunganisha mawakala watatu katika mtiririko wa kazi wa mfululizo ambao unaakisi uwasilishaji wa ujumbe wa A2A:

1. **CurrencyExchangeAgent** anapokea ombi la mtumiaji na kutoa mwongozo wa ubadilishaji wa sarafu.
2. **ActivityPlannerAgent** anapokea muktadha ulioboreshwa na kuongeza mapendekezo ya shughuli.
3. **TravelManagerAgent** anachanganya pembejeo zote mbili kuwa muhtasari wa mwisho wa safari.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuelewa A2A katika Uzalishaji

Katika mazingira ya uzalishaji itifaki ya A2A inafungua matukio yenye nguvu ya kuvuka-huduma:

| Uwezo | Maelezo |
|---|---|
| **Muunganisho kati ya mifumo** | Wakala uliojengwa kwa mfumo mmoja anaweza kumkabidhi kazi wakala aliyejengwa kwa mfumo mwingine wowote unaoendana na A2A, kuruhusu uunganishaji wa kweli kati ya mashirika. |
| **Mipaka ya huduma** | Wakala wanaweza kuishi katika microservices tofauti, mikoa ya wingu, au hata mashirika tofauti huku bado wakishirikiana bila mshono. |
| **Ugundaji wa wakati wa utekelezaji** | Mratibu anaweza kuulizia rejista ya Kadi ya Wakala wakati wa utekelezaji ili kupata mtaalamu anayefaa zaidi kwa kazi ndogo iliyotengwa. |
| **Utoaji wa mtiririko na arifa za kusukuma** | A2A inasaidia Server-Sent Events (SSE) kwa masasisho ya maendeleo kwa wakati halisi na arifa za kusukuma kwa kazi zinazochukua muda mrefu. |

Mtiririko wa kazi tuliojenga hapo juu ni toleo lililorahisishwa, ndani ya mchakato wa mfano huu. Katika utekelezaji wa kweli
kila wakala angeweka wazi kiunganishi cha HTTP, kuchapisha Kadi ya Wakala, na kuwasiliana
kupitia itifaki ya A2A JSON-RPC.


## Muhtasari

Katika somo hili ulijifunza:

1. **Protokali ya A2A ni nini** — kiwango wazi kwa ugunduzi kati ya mawakala, ujumbe,
   na usimamizi wa kazi.
2. **Jinsi ya kuunda mawakala maalum** — wakala wa kubadilisha sarafu, wakala mpangaji wa shughuli,
   na mratibu wa Meneja wa Kusafiri.
3. **Jinsi ya kuunganisha mawakala ndani ya mtiririko wa kazi** — kwa kutumia `WorkflowBuilder` kuunda mfano wa kupitisha ujumbe kwa mfululizo
   kati ya mawakala.
4. **Jinsi A2A inavyofanya kazi katika uzalishaji** — kuwezesha ushirikiano kati ya mifumo mbalimbali na huduma mbalimbali
   kwa ugunduzi unaobadilika na masasisho ya mtiririko.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Tamko la kutokuwa na dhamana:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuhakikisha usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au ukosefu wa usahihi. Nyaraka ya asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo chenye mamlaka. Kwa taarifa muhimu, inashauriwa kutumia tafsiri ya kitaalamu ya kibinadamu. Sisi hatuwajibiki kwa kutokuelewana au tafsiri zisizo sahihi zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
